# Poseable object

Example of displaying a ghost molecule that can be reposed in 3d space.

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.jupyter.utilities import make_id_generator
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: xr client cursors")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [3]:
from nanover.utilities.transforms import Transform

get_new_object_id = make_id_generator("object.")

OBJECTS = {}

class PoseableObject:
    objects: dict[str, "PoseableObject"] = {}

    @classmethod
    def make_basic(cls):
        object = cls(
            get_new_object_id(),
            Transform.identity(),
        )
        cls.objects[object.id] = object

    @classmethod
    def intersect_all(cls, point):
        for id, object in cls.objects.items():
            if object.contains_point(point):
                return object
        return None

    def __init__(self, id, transform):
        self.id = id
        self.transform = transform

        utilities.objects.update_shape(self.id, position=[0.0, 0.0, 0.0], color=[1.0, 0.0, 0.0, 1.0], size=1, parent=self.id)
        for z in range(2):
            for y in range(2):
                for x in range(2):
                    utilities.objects.update_shape(
                        f"{self.id}.{x}{y}{z}",
                        position=[x-.5, y-.5, z-.5],
                        color=[1.0-x*.5, 1.0-y*.5, 1.0-z*.5, 1.0],
                        size=.5,
                        parent=self.id,
                    )

    def update(self):
        utilities.transforms.update_transform(self.id, transform=self.transform)

    def contains_point(self, point):
        # todo: why does this feel inverted?
        point_local = self.transform.point_local_to_parent(point)
        utilities.notify_all(f"{np.linalg.norm(point_local)}")
        return np.linalg.norm(point_local) < 1.5

PoseableObject.make_basic()

In [4]:
import numpy as np
from nanover.jupyter import Mode
from MDAnalysis.lib.transformations import translation_matrix


OBJECT_RADIUS = 1
OBJECT_COLOR = [1.0, 0.0, 0.0, 1.0]
HOVER_COLOR = [1.0, 1.0, 1.0, 0.5]
CURSOR_RADIUS = 0

# mapping of cursor to currently grabbed object if any
CURSOR_GRABBED_OBJECT: dict[str, PoseableObject] = {}
# mapping of object id to object position
OBJECT_POSITIONS = {}


def intersect_objects(position):
    for object_id, object_pos in OBJECT_POSITIONS.items():
        if np.linalg.norm(np.subtract(position, object_pos)) < OBJECT_RADIUS + CURSOR_RADIUS:
            return object_id
    return None


class MoveObjectMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = PoseableObject.intersect_all(next_pos)
        available = hovered not in CURSOR_GRABBED_OBJECT.values()

        # grab hovered object if not already grabbed
        if button == "primary" and available:
            CURSOR_GRABBED_OBJECT[key] = hovered

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        # release grabbed
        if button == "primary":
            CURSOR_GRABBED_OBJECT.pop(key, None)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])

        # if this cursor has grabbed something, update its position
        grabbed = CURSOR_GRABBED_OBJECT.get(key, None)
        if grabbed is not None:
            grabbed.transform = Transform.from_parent_to_local_matrix(translation_matrix(next_pos))
            grabbed.update()

        # show/remove hover graphic is this cursor hovers something
        hovered = PoseableObject.intersect_all(next_pos)
        if hovered is None:
            utilities.objects.update_shape(f"hovered.{key}")
        else:
            utilities.objects.update_shape(f"hovered.{key}", position=[0.0, 0.0, 0.0], color=[1.0, 1.0, 0.0, .5], size=1.5, parent=hovered.id)


utilities.use_interaction_modes()
utilities.add_interaction_mode(MoveObjectMode, "move object", icon="✊")

In [5]:
utilities.show_logging()

Output()